# GPU Batching with TMol

TMol gains throughput by evaluating many structures together in one `PoseStack`. This notebook puts that capability near the start of the tutorial path and benchmarks scoring with synchronization placed correctly around CUDA timings.

> **Prerequisite.** Complete [Working with TMol](01_working_with_tmol.ipynb) first; this notebook reuses its `PoseStack`, CIF-input, and explicit-device vocabulary.

## Goals

- Build repeated batches with `PoseStackBuilder.from_poses`.
- Record enough hardware and software metadata to interpret timings.
- Measure latency, throughput, and peak CUDA memory after warmup.
- Provide a tiny CPU fallback and explain practical chunking.
- Distinguish TMol tensor batching from Chapter 16 job parallelism.

> **GPU recommended, not required.** CUDA is needed for representative acceleration and memory measurements. The same code uses tiny batch sizes on CPU so the notebook remains runnable.

## Setup

The benchmark fixes random seeds, discovers the checked-in 1UBQ fixture, and uses the same device for the pose, score function, and generated batches.

In [ ]:
from pathlib import Path
import platform
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from biotite.structure.io import load_structure

import tmol
from tmol import beta2016_score_function
from tmol.io.pose_stack_from_biotite import pose_stack_from_biotite
from tmol.pose.pose_stack_builder import PoseStackBuilder

SEED = 20260807
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
repo_root = Path.cwd()
if not (repo_root / "tmol/tests/data/cif/1UBQ.cif").exists():
    repo_root = Path(tmol.__file__).resolve().parents[1]
cif_path = repo_root / "tmol/tests/data/cif/1UBQ.cif"

atom_array = load_structure(str(cif_path), model=1, include_bonds=True)
protein_slice = atom_array[(atom_array.chain_id == "A") & (atom_array.res_id <= 20)]
single_pose = pose_stack_from_biotite(protein_slice, device, no_optH=True)
score_function = beta2016_score_function(device)


def show_table(frame):
    """Use sortable tables in rendered docs, with a pandas fallback."""
    try:
        from itables import show
    except ImportError:
        return display(frame)
    return show(frame)


print(f"benchmark device: {device}; input: {cif_path.name}")

## Record benchmark metadata first

Latency numbers are not portable without the PyTorch/TMol versions, operating system, CUDA runtime, and GPU identity. Record these before timing. Peak memory below is CUDA allocator memory, not total process or driver memory.

In [ ]:
metadata = {
    "platform": platform.platform(),
    "python": platform.python_version(),
    "tmol": tmol.__version__,
    "torch": torch.__version__,
    "device": str(device),
    "cuda_runtime": torch.version.cuda,
}
if device.type == "cuda":
    props = torch.cuda.get_device_properties(device)
    metadata.update(
        {
            "gpu_name": props.name,
            "compute_capability": f"{props.major}.{props.minor}",
            "gpu_memory_GiB": props.total_memory / 2**30,
        }
    )
metadata_frame = pd.DataFrame.from_dict(metadata, orient="index", columns=["value"])
show_table(metadata_frame.reset_index(names="property"))

## Build batches explicitly

`PoseStackBuilder.from_poses` concatenates compatible pose stacks and repacks their tensors on the requested device. Repeating the same pose is useful for a controlled throughput demonstration; real workloads normally batch distinct but chemistry-compatible structures.

In [ ]:
preview_batch = PoseStackBuilder.from_poses([single_pose] * 4, device)
print("single coords:", tuple(single_pose.coords.shape))
print("batch coords: ", tuple(preview_batch.coords.shape))
print("batch poses:  ", preview_batch.n_poses)

## Benchmark methodology

CUDA launches are asynchronous. A valid wall-clock measurement therefore:

1. renders the scorer before timing;
2. runs untimed warmup calls so lazy compilation and caches are not charged to steady state;
3. synchronizes before starting and after finishing each timed call; and
4. reports multiple repeats rather than one launch.

The CPU path uses the same function without CUDA synchronization and deliberately tiny sizes.

In [ ]:
def benchmark_scoring(batch_size, repeats=5, warmup=2):
    batch = PoseStackBuilder.from_poses([single_pose] * batch_size, device)
    scorer = score_function.render_whole_pose_scoring_module(batch)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    with torch.no_grad():
        for _ in range(warmup):
            scorer(batch.coords)
        if device.type == "cuda":
            torch.cuda.synchronize(device)

        elapsed = []
        for _ in range(repeats):
            if device.type == "cuda":
                torch.cuda.synchronize(device)
            start = perf_counter()
            scorer(batch.coords)
            if device.type == "cuda":
                torch.cuda.synchronize(device)
            elapsed.append(perf_counter() - start)

    median_seconds = float(np.median(elapsed))
    peak_memory = (
        torch.cuda.max_memory_allocated(device) / 2**20
        if device.type == "cuda"
        else np.nan
    )
    return {
        "batch_size": batch_size,
        "latency_ms": 1e3 * median_seconds,
        "throughput_poses_s": batch_size / median_seconds,
        "peak_cuda_MiB": peak_memory,
        "repeats": repeats,
    }

batch_sizes = [1, 4, 16, 64] if device.type == "cuda" else [1, 2, 4]
repeats = 5 if device.type == "cuda" else 2
benchmark_frame = pd.DataFrame(
    [benchmark_scoring(size, repeats=repeats) for size in batch_sizes]
)
show_table(benchmark_frame)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(
    benchmark_frame["batch_size"], benchmark_frame["latency_ms"], marker="o"
)
axes[0].set(xlabel="batch size", ylabel="median latency (ms)", title="Latency")
axes[1].plot(
    benchmark_frame["batch_size"],
    benchmark_frame["throughput_poses_s"],
    marker="o",
)
axes[1].set(
    xlabel="batch size", ylabel="poses / second", title="Scoring throughput"
)
for axis in axes:
    axis.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Expected observations.** Latency usually rises with batch size, while throughput improves until compute or memory bandwidth saturates. CPU fallback values are a correctness-oriented smoke run, not evidence about GPU scaling. First-call compilation can be much slower than the warm steady state and is intentionally excluded.

## CUDA-only capacity probe

The following cell is genuinely GPU-only and is labeled `gpu-only`. It demonstrates a larger batch and allocator memory. It is skipped cleanly on CPU.

In [ ]:
#| tags: [gpu-only]
# gpu-only: this capacity point is omitted on CPU.
if device.type != "cuda":
    print("Skipped: this cell requires CUDA.")
    gpu_capacity_point = None
else:
    gpu_capacity_point = benchmark_scoring(128, repeats=5, warmup=2)
    show_table(pd.DataFrame([gpu_capacity_point]))
gpu_capacity_point

## Memory and chunking

A larger batch is not always faster. `PoseStack` padding follows the largest member, and pairwise scoring intermediates can grow with both atom count and block count. Use the benchmark's peak-memory column, leave headroom for compilation and downstream tensors, and split a workload into chunks before allocator pressure causes an out-of-memory error.

A practical pattern is: choose a conservative per-device chunk size, build and score one chunk, immediately detach or transfer the small results you need, release chunk references, and continue. Grouping similarly sized structures reduces padding waste.

TMol has no built-in multi-GPU scheduler. One process should normally own one GPU; an external Slurm, Dask, Ray, or multiprocessing layer can shard independent chunks across devices.

## Rosetta comparison: two complementary levels of parallelism

TMol batching vectorizes compatible structures inside one process on one device. PyRosetta Chapter 16 mostly distributes independent jobs, trajectories, or protocols across processes and workers. Job distribution handles heterogeneous or long-running tasks; TMol batching amortizes kernels over tensor-compatible work. An outer scheduler can combine both ideas by assigning one TMol batch stream to each GPU.

Official Chapter 16 notebooks:

- [16.00 Running PyRosetta in Parallel](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.00-Running-PyRosetta-in-Parallel.ipynb)
- [16.01 distributed ddG/PSSM analysis](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.01-PyData-ddG-pssm.ipynb)
- [16.02 distributed miniprotein design](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.02-PyData-miniprotein-design.ipynb)
- [16.03 GNU Parallel via Slurm](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.03-GNU-Parallel-Via-Slurm.ipynb)
- [16.04 Dask delayed via Slurm](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.04-dask.delayed-Via-Slurm.ipynb)
- [16.05 distributed ligand docking](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.05-Ligand-Docking-dask.ipynb)
- [16.06 PyRosettaCluster simple protocol](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.06-PyRosettaCluster-Simple-protocol.ipynb)
- [16.07 reproduce a protocol](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.07-PyRosettaCluster-Reproduce-simple-protocol.ipynb)
- [16.08 multiple protocols](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.08-PyRosettaCluster-Multiple-protocols.ipynb)
- [16.09 multiple decoys](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.09-PyRosettaCluster-Multiple-decoys.ipynb)
- [16.10 ligand parameters](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.10-PyRosettaCluster-Ligand-params.ipynb)

The rendered [Chapter 16 index](https://rosettacommons.github.io/PyRosetta.notebooks/) provides context and setup instructions.

## Limitations

- This benchmark measures only repeated whole-pose scoring of one 20-residue layout; packing, minimization, and heterogeneous structures have different scaling.
- Warm steady-state timing intentionally excludes first-call compilation.
- Allocator peak memory is not total GPU memory usage.
- Repeated copies are pedagogical and may not reflect padding in a mixed real dataset.
- TMol does not provide a built-in distributed or multi-GPU runner; external orchestration is required.
- CPU fallback sizes are intentionally too small for accelerator conclusions.

## Exercises

1. Add interquartile latency alongside the median without timing scorer construction.
2. Increase CUDA batch sizes until throughput plateaus, stopping well before memory exhaustion.
3. Compare batches of similarly sized and mixed-length poses to quantify padding cost.
4. Implement an outer loop that scores a large list in conservative chunks and concatenates detached totals.
5. Design a Slurm array where each task selects one GPU and processes many TMol batches; keep random seeds and metadata per task.

## References

- [TMol repository](https://github.com/uw-ipd/tmol)
- [PyTorch CUDA semantics](https://pytorch.org/docs/stable/notes/cuda.html)
- [PyTorch benchmarking recipe](https://pytorch.org/tutorials/recipes/recipes/benchmark.html)
- [PyRosetta Chapter 16 index](https://rosettacommons.github.io/PyRosetta.notebooks/)
- [GNU Parallel via Slurm notebook](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.03-GNU-Parallel-Via-Slurm.ipynb)
- [Dask via Slurm notebook](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.04-dask.delayed-Via-Slurm.ipynb)